In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("IcebergLakehouse") \
    .master("spark://spark-master:7077") \
    .config("spark.jars", ",".join([
        "/opt/spark-extra-jars/iceberg-spark-runtime-3.5_2.12-1.6.1.jar",
        "/opt/spark-extra-jars/hadoop-aws-3.3.4.jar",
        "/opt/spark-extra-jars/aws-java-sdk-bundle-1.12.262.jar"
    ])) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.iceberg.type", "hadoop") \
    .config("spark.sql.catalog.iceberg.warehouse", "s3a://warehouse/") \
    .config("spark.sql.defaultCatalog", "iceberg") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

25/09/20 04:22:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
from minio import Minio

In [3]:
client = Minio(
    "minio:9000",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False
)

In [4]:
bucket = 'warehouse'
if not client.bucket_exists(bucket):
    client.make_bucket(bucket)

In [5]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS iceberg.demo.nyc_taxis (
        vendor_id BIGINT,
        trip_id STRING,
        trip_distance DOUBLE,
        fare_amount DOUBLE,
        store_and_fwd_flag STRING,
        trip_start_timestamp TIMESTAMP  -- Added missing column
    ) USING iceberg
    PARTITIONED BY (days(trip_start_timestamp))  -- Hidden partitioning for flexibility
    TBLPROPERTIES (
        'format-version' = '2'  -- Enables v2 features like row deletes
    )
    LOCATION 's3a://warehouse/demo/nyc_taxis'
""")

25/09/20 04:22:32 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


DataFrame[]

In [6]:
spark.sql("SHOW TABLES IN iceberg.demo").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|     demo|nyc_taxis|      false|
+---------+---------+-----------+



In [7]:
spark.sql("DESCRIBE iceberg.demo.nyc_taxis").show()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|           vendor_id|              bigint|   NULL|
|             trip_id|              string|   NULL|
|       trip_distance|              double|   NULL|
|         fare_amount|              double|   NULL|
|  store_and_fwd_flag|              string|   NULL|
|trip_start_timestamp|           timestamp|   NULL|
|                    |                    |       |
|      # Partitioning|                    |       |
|              Part 0|days(trip_start_t...|       |
+--------------------+--------------------+-------+



In [8]:
spark.sql("""
    INSERT INTO iceberg.demo.nyc_taxis
    VALUES (1, 'trip001', 2.5, 10.0, 'N', CAST('2025-09-20 10:00:00' AS TIMESTAMP))
""")
spark.sql("SELECT * FROM iceberg.demo.nyc_taxis").show()

+---------+-------+-------------+-----------+------------------+--------------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|trip_start_timestamp|
+---------+-------+-------------+-----------+------------------+--------------------+
|        1|trip001|          2.5|       10.0|                 N| 2025-09-20 10:00:00|
+---------+-------+-------------+-----------+------------------+--------------------+



In [9]:
spark.createDataFrame([(1, "Alice"), (2, "Bob")], ["id", "name"]).show()

+---+-----+
| id| name|
+---+-----+
|  1|Alice|
|  2|  Bob|
+---+-----+



In [10]:
file_path = "data/raw/fhvhv/fhvhv_tripdata_2021-01.csv.gz"  # file inside shared /data
object_name = "data/raw/fhvhv/fhvhv_tripdata_2021-01.csv.gz"

client.fput_object(bucket, object_name, file_path)

print(f"Uploaded {file_path} → {bucket}/{object_name}")

Uploaded data/raw/fhvhv/fhvhv_tripdata_2021-01.csv.gz → warehouse/data/raw/fhvhv/fhvhv_tripdata_2021-01.csv.gz


In [11]:
# List all buckets
buckets = client.list_buckets()
for bucket in buckets:
    print(bucket.name, bucket.creation_date)

warehouse 2025-09-20 04:22:29.725000+00:00


In [12]:
for obj in client.list_objects("warehouse", recursive=True):
    print(obj.bucket_name, obj.object_name, obj.size)

warehouse data/raw/fhvhv/fhvhv_tripdata_2021-01.csv.gz 129967421
warehouse demo/nyc_taxis/data/trip_start_timestamp_day=2025-09-20/00000-1-3d642d2b-c853-48c3-86a7-c056afbb6ba7-0-00001.parquet 1806
warehouse demo/nyc_taxis/metadata/7a980450-c902-4ea8-b109-6f124adc5177-m0.avro 7317
warehouse demo/nyc_taxis/metadata/snap-423891401702399540-1-7a980450-c902-4ea8-b109-6f124adc5177.avro 4450
warehouse demo/nyc_taxis/metadata/v1.metadata.json 1596
warehouse demo/nyc_taxis/metadata/v2.metadata.json 2821
warehouse demo/nyc_taxis/metadata/version-hint.text 1


In [13]:
from pyspark.sql import types
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [14]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv("s3a://warehouse/data/raw/fhvhv/fhvhv_tripdata_2021-01.csv.gz")

In [15]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   NULL|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   NULL|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   NULL|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [16]:
df = df.repartition(24)

In [17]:
df.write.parquet('s3a://warehouse/data/pq/fhvhv/')

In [18]:
for obj in client.list_objects("warehouse", recursive=True):
    print(obj.bucket_name, obj.object_name, obj.size)

warehouse data/pq/fhvhv/_SUCCESS 0
warehouse data/pq/fhvhv/part-00000-922002f5-8f6a-432e-9cf6-277f126f1b29-c000.snappy.parquet 9843343
warehouse data/pq/fhvhv/part-00001-922002f5-8f6a-432e-9cf6-277f126f1b29-c000.snappy.parquet 9838623
warehouse data/pq/fhvhv/part-00002-922002f5-8f6a-432e-9cf6-277f126f1b29-c000.snappy.parquet 9832345
warehouse data/pq/fhvhv/part-00003-922002f5-8f6a-432e-9cf6-277f126f1b29-c000.snappy.parquet 9846376
warehouse data/pq/fhvhv/part-00004-922002f5-8f6a-432e-9cf6-277f126f1b29-c000.snappy.parquet 9832110
warehouse data/pq/fhvhv/part-00005-922002f5-8f6a-432e-9cf6-277f126f1b29-c000.snappy.parquet 9839000
warehouse data/pq/fhvhv/part-00006-922002f5-8f6a-432e-9cf6-277f126f1b29-c000.snappy.parquet 9842041
warehouse data/pq/fhvhv/part-00007-922002f5-8f6a-432e-9cf6-277f126f1b29-c000.snappy.parquet 9841234
warehouse data/pq/fhvhv/part-00008-922002f5-8f6a-432e-9cf6-277f126f1b29-c000.snappy.parquet 9836032
warehouse data/pq/fhvhv/part-00009-922002f5-8f6a-432e-9cf6-277f12

In [19]:
df = spark.read.parquet('s3a://warehouse/data/pq/fhvhv/')

In [20]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [21]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
    .filter(df.hvfhs_license_num == 'HV0003').show()

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-01 16:47:20|2021-01-01 16:58:28|          50|         163|
|2021-01-05 02:00:14|2021-01-05 02:19:39|          48|          95|
|2021-01-02 00:34:43|2021-01-02 00:45:38|          63|          77|
|2021-01-02 16:20:11|2021-01-02 16:56:36|          63|         244|
|2021-01-24 16:00:53|2021-01-24 16:07:40|         210|         165|
|2021-01-16 19:35:17|2021-01-16 19:50:20|         113|         143|
|2021-01-01 11:15:17|2021-01-01 11:24:55|         231|         148|
|2021-01-19 12:05:32|2021-01-19 12:33:46|         228|         210|
|2021-01-17 13:54:52|2021-01-17 14:07:03|          39|          61|
|2021-01-30 18:03:33|2021-01-30 18:23:17|          42|         250|
|2021-01-16 12:36:55|2021-01-16 13:03:23|         131|         265|
|2021-01-30 23:07:14|2021-01-30 23:27:34|       